In [ ]:
!pip install pandas pgmpy

In [16]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

In [22]:
df = pd.read_csv("heart.csv")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB


In [23]:
df = df[['age', 'cp', 'exang', 'target']].dropna()
print(df.head())

   age  cp  exang  target
0   52   0      0       0
1   53   0      1       0
2   70   0      1       0
3   61   0      0       0
4   62   0      0       0


In [24]:
df['age'] = pd.cut(
    df['age'],
    bins=[0, 40, 55, 100],
    labels=['Young', 'Middle', 'Senior']
)

df = df.astype(str)

print(df.head())

      age cp exang target
0  Middle  0     0      0
1  Middle  0     1      0
2  Senior  0     1      0
3  Senior  0     0      0
4  Senior  0     0      0


In [25]:
model = DiscreteBayesianNetwork([
    ('age', 'target'),
    ('cp', 'target'),
    ('exang', 'target')
])

print("Bayesian Network Created")

Bayesian Network Created


In [26]:
model.fit(df)

print("Model Trained Successfully")

Model Trained Successfully


In [27]:
inference = VariableElimination(model)

result = inference.query(
    variables=['target'],
    evidence={
        'age': 'Middle',
        'cp': '2',
        'exang': '0'
    },
    show_progress=False
)

print(result)

+-----------+---------------+
| target    |   phi(target) |
+===========+===============+
| target(0) |        0.1417 |
+-----------+---------------+
| target(1) |        0.8583 |
+-----------+---------------+


In [28]:
prob = result.values

print("No Heart Disease:", round(prob[0] * 100, 2), "%")
print("Heart Disease:", round(prob[1] * 100, 2), "%")

if prob[1] > prob[0]:
    print("Heart Disease Detected")
else:
    print("No Heart Disease Detected")

No Heart Disease: 14.17 %
Heart Disease: 85.83 %
Heart Disease Detected
